In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('ohlc_raw.csv')
print(f"Raw shape: {df.shape[0]:,} rows × {df.shape[1]} cols\n")

# ── 1. Type enforcement ───────────────────────────────────────────────────────
df['date']    = pd.to_datetime(df['date'], format='%Y-%m-%d')
df['company'] = df['company'].astype(str).str.strip()
df['ticker']  = df['ticker'].astype(str).str.strip().str.upper()
df[['open','high','low','close']] = df[['open','high','low','close']].astype('float64')
df['volume']  = pd.to_numeric(df['volume'], errors='coerce')
print("[1] Types enforced — date→datetime, prices→float64, volume→numeric")

# ── 2. Duplicates ─────────────────────────────────────────────────────────────
n_dupes = df.duplicated(subset=['date','company']).sum()
df = df.drop_duplicates(subset=['date','company'], keep='first').reset_index(drop=True)
print(f"[2] Duplicates: {n_dupes} found and dropped")

# ── 3. Sector column (metadata enrichment) ────────────────────────────────────
_sector_order = [
    ("Oil & Gas — OMCs",        ["HPCL","BPCL","Indian Oil","Indraprastha Gas","Mahanagar Gas","Gujarat Gas"]),
    ("Oil & Gas — Upstream",    ["ONGC","Oil India","Vedanta"]),
    ("Defence",                 ["HAL","BDL","BEL","BEML","Mazagon Dock","GRSE","Cochin Shipyard","Data Patterns"]),
    ("Aviation",                ["IndiGo","SpiceJet","Air India"]),
    ("Paints",                  ["Asian Paints","Berger Paints","Kansai Nerolac","Indigo Paints"]),
    ("Tyres",                   ["MRF","Apollo Tyres","CEAT","Balkrishna Ind"]),
    ("Pharma",                  ["Sun Pharma","Dr Reddys","Divis Labs","Cipla"]),
    ("Shipping & Ports",        ["Adani Ports","Shipping Corp","Great Eastern","Essar Shipping"]),
    ("Capital Goods",           ["L&T","Thermax","KEC International"]),
    ("Gold-Linked",             ["Titan","Kalyan Jewellers","Muthoot Finance","Manappuram"]),
    ("Chemicals & Fertilisers", ["Coromandel","Deepak Nitrite","GNFC","Tata Chemicals"]),
    ("IT & Tech",               ["Infosys","Wipro","HCL Tech"]),
]
SECTOR_MAP = {co: sector for sector, companies in _sector_order for co in companies}
df['sector'] = df['company'].map(SECTOR_MAP)
unmapped = df['sector'].isna().sum()
print(f"[3] Sector column added — {df['sector'].nunique()} sectors"
      + (f", {unmapped} rows unmapped" if unmapped else ""))

# ── 4. Missing values ─────────────────────────────────────────────────────────
null_counts = df[['open','high','low','close','volume']].isna().sum()
print(f"[4] Null counts: {dict(null_counts)}")

partial   = df['close'].isna() & df['volume'].notna()
fully_null = df['close'].isna() & df['volume'].isna()
print(f"    Partial-day rows (OHLC null, volume present): {partial.sum()} — dropping")
print(f"    Fully null rows (no OHLC, no volume):         {fully_null.sum()} — dropping")

df = df.dropna(subset=['open','high','low','close','volume']).reset_index(drop=True)
print(f"    Remaining: {len(df):,} rows")

# ── 5. Precision normalisation ────────────────────────────────────────────────
for col in ['open','high','low','close']:
    df[col] = df[col].round(2)
df['volume'] = df['volume'].astype(np.int64)
print("[5] Prices rounded to 2 d.p. (IEEE 754 noise removed), volume cast to int64")

# ── 6. OHLC logical constraint validation ─────────────────────────────────────
checks = {
    'high < low'     : (df['high'] < df['low']).sum(),
    'close > high'   : (df['close'] > df['high']).sum(),
    'close < low'    : (df['close'] < df['low']).sum(),
    'open > high'    : (df['open'] > df['high']).sum(),
    'open < low'     : (df['open'] < df['low']).sum(),
    'negative price' : ((df[['open','high','low','close']] <= 0).any(axis=1)).sum(),
    'zero volume'    : (df['volume'] <= 0).sum(),
}
print("[6] OHLC logical validation:")
for rule, count in checks.items():
    print(f"    {'⚠' if count else '✓'} {rule}: {count} violations")

# ── 7. Outlier detection (documented only — data unchanged) ───────────────────
df_sorted = df.sort_values(['company','date'])
daily_return = df_sorted.groupby('company')['close'].pct_change() * 100
shocks = daily_return[daily_return.abs() > 10]
print(f"[7] Outlier report (data NOT modified — stock outliers are real events):")
print(f"    Price shocks (|daily return| > 10%): {len(shocks)}")
for idx, val in shocks.items():
    row = df_sorted.loc[idx]
    print(f"      {str(row['date'].date())}  {row['company']:20s}  {val:+.2f}%")

# ── 8. Export ─────────────────────────────────────────────────────────────────
df = df[['date','company','ticker','sector','open','high','low','close','volume']]
df = df.sort_values(['company','date']).reset_index(drop=True)
df.to_csv('ohlc_clean.csv', index=False)
print(f"\nSaved: ohlc_clean.csv — {df.shape[0]:,} rows × {df.shape[1]} cols")

Raw shape: 1,400 rows × 8 cols

[1] Types enforced — date→datetime, prices→float64, volume→numeric
[2] Duplicates: 0 found and dropped
[3] Sector column added — 12 sectors
[4] Null counts: {'open': np.int64(131), 'high': np.int64(131), 'low': np.int64(131), 'close': np.int64(131), 'volume': np.int64(84)}
    Partial-day rows (OHLC null, volume present): 47 — dropping
    Fully null rows (no OHLC, no volume):         84 — dropping
    Remaining: 1,269 rows
[5] Prices rounded to 2 d.p. (IEEE 754 noise removed), volume cast to int64
[6] OHLC logical validation:
    ✓ high < low: 0 violations
    ✓ close > high: 0 violations
    ✓ close < low: 0 violations
    ✓ open > high: 0 violations
    ✓ open < low: 0 violations
    ✓ negative price: 0 violations
    ✓ zero volume: 0 violations
[7] Outlier report (data NOT modified — stock outliers are real events):
    Price shocks (|daily return| > 10%): 4
      2026-04-01  Cochin Shipyard       +12.20%
      2026-04-10  Essar Shipping        +11.9